In [2]:
# %pip install pandas matplotlib seaborn
import requests
import zipfile
import io
import os
import pandas as pd
import duckdb
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path



In [ ]:
# Test Download Data 1 

# Create data directory
Path("data_test").mkdir(exist_ok=True)

# Download one month
url = "https://divvy-tripdata.s3.amazonaws.com/202303-divvy-tripdata.zip"
zip_path = "data_test/202303-divvy-tripdata.zip"

print("Downloading data...")
response = requests.get(url)
with open(zip_path, 'wb') as f:
    f.write(response.content)

# Extract
print("Extracting...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("data_test/")
    
print("Download complete!")

In [ ]:
# Test Download Data 2

# Create a data directory
os.makedirs('data_test', exist_ok=True)

# Divvy data URL 
url = "https://divvy-tripdata.s3.amazonaws.com/202403-divvy-tripdata.zip"
zip_name = "202403-divvy-tripdata.zip"

print("Downloading data...")
r = requests.get(url)
z = zipfile.ZipFile(io.BytesIO(r.content))

print("Extracting...")
z.extractall("data_test")
print("Done! Files in folder:", os.listdir("data_test"))

In [ ]:
# Find a CSV file
csv_file = list(Path("data_test").glob("*.csv"))[0]
print(f"Loading: {csv_file}")

# Load data
df = pd.read_csv(csv_file)

print(f"Shape: {df.shape}")
df.head()

In [ ]:
# From Ingestion - Now we have a data lake with parquet files. Let's see what files we have.
os.listdir("../data/raw")

In [ ]:
# See the parquet file
par_file = list(Path("../data/raw").glob("*.parquet"))[0]
print(f"Loading: {par_file}")

# Load data
df = pd.read_parquet(par_file)

print(f"Shape: {df.shape}")
df.head()

In [ ]:
# Load and Inspect
print(list(Path("../data/raw").glob("*.parquet")))



In [5]:
con = duckdb.connect("/workspaces/dezoomcamp-divvy-chicago-dbt/data/warehouse.duckdb")

In [4]:
con.close()

In [7]:
df_dupes = con.execute("""
    SELECT
        ride_id,
        COUNT(*) as cnt
    FROM read_parquet('/workspaces/dezoomcamp-divvy-chicago-dbt/data/raw/*.parquet')
    GROUP BY ride_id
    HAVING COUNT(*) > 1
    ORDER BY cnt DESC
""").df()

df_dupes.head()

,ride_id,cnt
0,3B5CE4D8B3EE6ED8,2
1,60B4DDFF369931B2,2
2,B1B9E66D2E7C383D,2
3,456B53D1210C6052,2
4,FEA150E7A56F187E,2


In [8]:
df_details = con.execute("""
    SELECT *
    FROM read_parquet('/workspaces/dezoomcamp-divvy-chicago-dbt/data/raw/*.parquet')
    WHERE ride_id IN (
        SELECT ride_id
        FROM read_parquet('/workspaces/dezoomcamp-divvy-chicago-dbt/data/raw/*.parquet')
        GROUP BY ride_id
        HAVING COUNT(*) > 1
    )
    ORDER BY ride_id, started_at
""").df()

df_details.head(20)

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,011C8EF97AB0F30D,classic_bike,2024-05-31 19:45:38,2024-06-01 20:45:33,Clifton Ave & Armitage Ave,TA1307000163,None,None,41.918216,-87.656936,NaN,NaN,casual
1,011C8EF97AB0F30D,classic_bike,2024-05-31 19:45:38.037,2024-06-01 20:45:33.862,Clifton Ave & Armitage Ave,TA1307000163,None,None,41.918216,-87.656936,NaN,NaN,casual
2,01406457A85B0AFF,electric_bike,2024-05-31 23:54:59,2024-06-01 00:01:47,None,None,Damen Ave & Chicago Ave,13132,41.890000,-87.670000,41.895769,-87.677220,member
3,01406457A85B0AFF,electric_bike,2024-05-31 23:54:59.194,2024-06-01 00:01:47.626,None,None,Damen Ave & Chicago Ave,13132,41.890000,-87.670000,41.895769,-87.677220,member
4,02606FBC7F8537EE,classic_bike,2024-05-31 17:55:01,2024-06-01 18:54:53,Pine Grove Ave & Waveland Ave,TA1307000150,None,None,41.949473,-87.646453,NaN,NaN,casual
5,02606FBC7F8537EE,classic_bike,2024-05-31 17:55:01.635,2024-06-01 18:54:53.970,Pine Grove Ave & Waveland Ave,TA1307000150,None,None,41.949473,-87.646453,NaN,NaN,casual
6,0354FD0756337B59,electric_bike,2024-05-31 23:34:36,2024-06-01 00:14:29,None,None,None,None,41.970000,-87.660000,41.960000,-87.660000,casual
7,0354FD0756337B59,electric_bike,2024-05-31 23:34:36.273,2024-06-01 00:14:29.238,None,None,None,None,41.970000,-87.660000,41.960000,-87.660000,casual
8,048C715F1DE0D8C0,electric_bike,2024-05-31 23:53:44,2024-06-01 00:12:26,None,None,None,None,41.890000,-87.660000,41.890000,-87.670000,casual
9,048C715F1DE0D8C0,electric_bike,2024-05-31 23:53:44.401,2024-06-01 00:12:26.776,None,None,None,None,41.890000,-87.660000,41.890000,-87.670000,casual


In [ ]:
df_see = con.execute("""
    WITH ranked AS (
        SELECT *,
            ROW_NUMBER() OVER (
                PARTITION BY ride_id
                ORDER BY started_at
            ) AS rn
        FROM read_parquet('/workspaces/dezoomcamp-divvy-chicago-dbt/data/raw/*.parquet')
    )
    SELECT *
    FROM ranked
    WHERE rn = 1
""").df()

df_see.head(20)

In [ ]:
df = con.execute("""
    SELECT *
    FROM read_parquet('/workspaces/dezoomcamp-divvy-chicago-dbt/data/raw/*.parquet')
    LIMIT 10
""").df()

df.head()

In [ ]:
con.execute("""
    SELECT COUNT(*) 
    FROM (
        SELECT ride_id
        FROM (
            SELECT *,
                ROW_NUMBER() OVER (
                    PARTITION BY ride_id
                    ORDER BY started_at
                ) AS rn
            FROM read_parquet('/workspaces/dezoomcamp-divvy-chicago-dbt/data/raw/*.parquet')
        )
        WHERE rn = 1
        GROUP BY ride_id
        HAVING COUNT(*) > 1
    )
""").fetchall()

## Inspect the dataframe

In [ ]:
# Basic Info
df.info()



In [ ]:
# Summary Statistics
df.describe()

In [ ]:
# Data Quality Checks
print("Missing values:")
print(df.isnull().sum())

print("\n" + "="*50)
print("Unique ride types:")
print(df['rideable_type'].value_counts())

print("\n" + "="*50)
print("Member vs Casual:")
print(df['member_casual'].value_counts())

In [ ]:
# DateTime Analysis
df['started_at'] = pd.to_datetime(df['started_at'])
df['ended_at'] = pd.to_datetime(df['ended_at'])

# Calculate trip duration
df['trip_duration_min'] = (df['ended_at'] - df['started_at']).dt.total_seconds() / 60

# Add time features
df['hour'] = df['started_at'].dt.hour
df['day_of_week'] = df['started_at'].dt.day_name()

print("Trip duration stats (minutes):")
print(df['trip_duration_min'].describe())



In [ ]:
# Visualizations
plt.style.use('seaborn-v0_8-darkgrid')
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Trips by hour
df['hour'].value_counts().sort_index().plot(kind='bar', ax=axes[0,0])
axes[0,0].set_title('Trips by Hour of Day')
axes[0,0].set_xlabel('Hour')

# Trips by day of week
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
df['day_of_week'].value_counts()[day_order].plot(kind='bar', ax=axes[0,1])
axes[0,1].set_title('Trips by Day of Week')

# Trip duration distribution (filtered outliers)
df[df['trip_duration_min'] < 60]['trip_duration_min'].hist(bins=50, ax=axes[1,0])
axes[1,0].set_title('Trip Duration Distribution (< 60 min)')
axes[1,0].set_xlabel('Minutes')

# Member vs Casual
df['member_casual'].value_counts().plot(kind='pie', ax=axes[1,1], autopct='%1.1f%%')
axes[1,1].set_title('Member vs Casual Riders')

plt.tight_layout()
plt.show()

In [ ]:
# Station Analysis
print("Top 10 Start Stations:")
print(df['start_station_name'].value_counts().head(10))

print("\n" + "="*50)
print("Top 10 End Stations:")
print(df['end_station_name'].value_counts().head(10))

In [ ]:
# Data Quality Issues to Address
print("Data Quality Issues Found:")
print(f"1. Missing station names: {df['start_station_name'].isnull().sum()}")
print(f"2. Negative trip durations: {(df['trip_duration_min'] < 0).sum()}")
print(f"3. Very long trips (>24h): {(df['trip_duration_min'] > 1440).sum()}")
print(f"4. Very short trips (<1min): {(df['trip_duration_min'] < 1).sum()}")